In [10]:
#%run func_defs.ipynb

%run NN_defs.ipynb

import time, os, random
seed = 42

start_new_nn = 1
ns = 300 ;
nz = 640; N_samples= 8100
Lz=10 ; tf = 20; 
k1=1.0; k2= 2.0 
lr = .01
wd = .0001
epochs = 10000000
betas = (0.995, 0.999)
tol = 5e-08
#nodes = 32
#out_file= f'./model/2layer_nz{nz}_tf{tf}.pt'
out_file= f'./model/2layer_nz{nz}_tf{tf}_random.pt'
dat_file = f'./data/2layer_k1{k1}_k2{k2}_nz{nz}_random_N{N_samples}.pt'
#dat_file = f'./data/2layer_k1{k1}_k2{k2}_ns{ns}.pt'

d_in= 4
#2layer_k11.0_k22.0_nz640.pt

In [11]:
# set a dictionary of parameters
params_def = {'d_in':4, 'd_out':1, 'layers':4, 'nodes':64,
    'tol':0.5e-6, 'epochs':1000000, 'seg_size':1000, 'epoch_0':0,
    'lr':0.005, 'eps':1e-10, 'betas':(0.995, 0.999), 'wd':1e-4,
    'L':10, 'N':100000, 'N_test':100000,
    'start_new_nn':1, 'dat_file':f'./data/tm_in.pt',
    'out_file':f'./data/tm_out.pt'} 
for name in params_def: # using existing unless not existent
    if name not in globals():
        globals()[name] = params_def[name]

sec_main = """
# Load the training data
trn_dat = torch.load(dat_file, weights_only=True)

device = torch.device('cuda')

stau = trn_dat['stau']
W = trn_dat['W']
kratio = trn_dat['kratio']
z_disv = trn_dat['zdis']


train = torch.stack([stau[:,0],stau[:,1], kratio, z_disv], dim=-1)

xtrain = train.to(torch.float64)
ytrain = W.to(torch.float64)

print(train.shape)
print(W.shape)
if start_new_nn == 1:
    model, optimizer = build_nn() # build a new model
    model = model.to(device)
    del start_new_nn
else:
    model, optimizer = rebuild_nn() # rebuild the model from saved data
    model = model.to(device)
    for state in optimizer.state.values():
        for k, v in state.items():
            if isinstance(v, torch.Tensor):
                state[k] = v.to(device)
print(f'neural network model is on {next(model.parameters()).device}')
print_params() # print parameters
#
trained_model = train_nn() # train the model
for key in trained_model: # unpack trained_model
    globals()[key] = trained_model[key]
# test_result = test_and_plotting() # test the model and plot results
# for key in test_result: # unpack test_result
   # globals()[key] = test_result[key]
#
print_params()
[print(msg) for msg in msg_train]
# [print(msg) for msg in msg_test
save_model_vars() # save the model and variables in a dictionary
"""


In [12]:
# run the code unless func_def_only ==1

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

exec(sec_main)


torch.Size([8100, 4])
torch.Size([8100, 1])
neural network model is on cuda:0
Structure of neural network:  Net(
  (layers): ModuleList(
    (0): Linear(in_features=4, out_features=64, bias=True)
    (1-3): 3 x Linear(in_features=64, out_features=64, bias=True)
  )
  (out): Linear(in_features=64, out_features=1, bias=True)
)
Optimizer:   AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.995, 0.999)
    capturable: False
    differentiable: False
    eps: 1e-10
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    weight_decay: 0.0001
)
parameters: d_in = 4, d_out = 1, layers = 4, nodes = 64
    tol = 5.0000e-08, epochs = 10000000, seg_size = 1000, epoch_0 = 0
    N_train = 100000, N_test = 100000
    output file: ./model/2layer_nz640_tf20_random.pt

i = 0, loss = 6.4367e-01 loss_curr_min = 6.4367e-01
Elapsed time = 0.00s,  CA time = Fri Jul  3 16:22:53 2026
i = 1000, loss = 2.9415e-03 loss_curr_min = 2.7994e-03
Elapsed time = 2.85s,  CA time = Fri Jul  3 16:22